# 03 Gold: analysis views and figure tables

Gold is a view layer over the cleaned Silver data (`silver.sor`), restricted to the analysis set: Germany
in the territorial scope, content posted from 2025-01-01 to 2025-06-30, statement submitted before
2025-07-01. Nothing large is materialised.
Every table that feeds a figure is frozen as a small CSV file in `results/`, so the figures reported
in the thesis remain stable between drafts. Each table is followed by a descriptive statement of what it
shows; interpretation follows in the discussion.

## Setup

In [1]:
from config import SCHEMA_BREAK, connect

import time
from pathlib import Path

import pandas as pd

con = connect()
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 90)

GOLD_OUT = Path("results")
GOLD_OUT.mkdir(parents=True, exist_ok=True)


def freeze(df: pd.DataFrame, name: str) -> pd.DataFrame:
    """Write a figure table to results/<name>.csv and return it unchanged."""
    df.to_csv(GOLD_OUT / f"{name}.csv", index=False)
    return df

## 1. Analytical decisions

**Weekly time unit.** Both platforms are analysed in ISO weeks starting on Monday. X averages 730
statements per day, and its daily series is unstable (coefficient of variation 0.33 in April 2025). Both
platforms show weekday effects: across weekdays, TikTok's mean daily volume ranges from 0.94 to 1.08 of
its overall daily mean and X's from 0.87 to 1.15. Weekly aggregation removes these effects. A daily
series for TikTok is kept as a supplementary view (table 9).

**Moderation intensity as a share.** TikTok's total weekly volume more than doubles between January and
March 2025, from 19.5 million statements in the week of 6 January to 45.7 million in the week of
10 March, independently of the election category. Intensity is therefore measured as the share of
statements in the category `STATEMENT_CATEGORY_NEGATIVE_EFFECTS_ON_CIVIC_DISCOURSE_OR_ELECTIONS` among all
statements of the same platform in the same week or period, which controls for this growth.

**Periods.** Five periods are defined on `application_date`: first baseline 2025-01-01 to 2025-01-19,
campaign 2025-01-20 to 2025-02-23, post-election 2025-02-24 to 2025-03-31, transition 2025-04-01 to
2025-04-30, second baseline 2025-05-01 to 2025-06-30. The election-category share rises in the week of
20 January 2025 and returns to its starting level in May. Two baselines, before and after the event,
are used, and their agreement (3.085% and 2.967%, table 6) is reported. The period label is derived in
this layer only; the Silver layer carries no period attribute, so a change to the periods requires no
rebuild. Because the analysis set contains only statements submitted before 2025-07-01, every statement falls into one of the five periods.

**Time axis.** The time axis is `application_date`, the date on which the measure was applied, as in
Trujillo et al. (2025), Groesch et al. (2026) and Shahi et al. (2025). The choice has little effect in
this data: 96.87% of TikTok statements and all X statements have an application delay of zero days.

**Granularity.** The strict count `n_filled_strictly_optional` is the primary measure. The conditionally
required attributes are excluded from it because their presence depends on `decision_ground`, which
differs systematically between the platforms. The full count `n_filled_optional_fields` serves as the
robustness check.

**X in the event analysis.** X reports no statement in the election category. It enters the event
analysis through volume and granularity only.

**Modality.** Modality is examined within TikTok, which holds the platform constant. A comparison of content types within X is not possible: X classifies 99.995% of its statements as synthetic media and reports no text, video or image content (table 3b), and only six of its statements are decided fully automatically (table 2).

**Measure types.** The coded measure attributes are combined with the free text in
`decision_visibility_other`. Each free-text value is assigned an action class and a measure type
(appendix A). Where a statement carries several restriction types, only the hardest is counted, in the
order account, monetary, provision of service, visibility, following Groesch et al. (2026). The
non-publication of an advertisement is kept as a separate type, since it is not a reduction of reach.

**View layer with frozen tables.** Gold consists of views over the Silver parquet. Simple aggregations
take seconds (rows per platform and month: 1.5 s); the five passes behind the tables below take several
minutes each (section 2.1). Frozen tables keep the figures reported in the thesis stable.

**Explanation attributes excluded.** `decision_facts`, `illegal_content_explanation` and
`incompatible_content_explanation` are not used in the analysis. They are retained in Silver and
excluded from the Gold views.

**Week and period boundaries.** Three period boundaries do not coincide with ISO week boundaries:
2025-03-31 is a Monday, 2025-04-01 a Tuesday and 2025-05-01 a Thursday. Weeks spanning two periods are
labelled with both (`mixed: a/b`) in the weekly tables. Period totals (table 6) are computed from days,
not weeks, and are exact. The first ISO week (starting 2024-12-30) and the last (starting 2025-06-30)
contain fewer than seven days inside the window; the number of days is reported per week.

## 2. Gold views

In [2]:
ELECTION = "STATEMENT_CATEGORY_NEGATIVE_EFFECTS_ON_CIVIC_DISCOURSE_OR_ELECTIONS"
PERIODS = [
    ("baseline_a", "2025-01-01", "2025-01-19"),
    ("campaign", "2025-01-20", "2025-02-23"),
    ("post_election", "2025-02-24", "2025-03-31"),
    ("transition", "2025-04-01", "2025-04-30"),
    ("baseline_b", "2025-05-01", "2025-06-30"),
]
period_case = "CASE " + " ".join(
    f"WHEN application_date::DATE BETWEEN DATE '{a}' AND DATE '{b}' THEN '{p}'" for p, a, b in PERIODS
) + " ELSE 'outside' END"

con.execute("CREATE SCHEMA IF NOT EXISTS gold")
con.execute("CREATE OR REPLACE VIEW gold.periods AS SELECT * FROM (VALUES " + ", ".join(
    f"('{p}', DATE '{a}', DATE '{b}')" for p, a, b in PERIODS) + ") t(period, start_date, end_date)")

# One row per statement, as in Silver; the explanation attributes are excluded.
con.execute(f"""
    CREATE OR REPLACE VIEW gold.sor AS
    SELECT * EXCLUDE (decision_facts, illegal_content_explanation, incompatible_content_explanation),
           application_date::DATE                        AS app_day,
           date_trunc('week', application_date)::DATE   AS week_start,
           {period_case}                                 AS period,
           category = '{ELECTION}'                       AS is_election
    FROM silver.sor
    WHERE content_date < DATE '{SCHEMA_BREAK}' AND created_at < TIMESTAMP '{SCHEMA_BREAK}'
""")

con.sql("SELECT * FROM gold.periods").df()

,period,start_date,end_date
0,baseline_a,2025-01-01,2025-01-19
1,campaign,2025-01-20,2025-02-23
2,post_election,2025-02-24,2025-03-31
3,transition,2025-04-01,2025-04-30
4,baseline_b,2025-05-01,2025-06-30


### 2.1 Aggregation passes

Four scans of the Gold view produce small aggregates from which the tables are derived; a fifth follows
the measure classification in 2.2. Each pass reads the analysis set once; its runtime is printed.

In [3]:
def scan(label, sql):
    t = time.time()
    df = con.sql(sql).df()
    print(f"{label}: {len(df):,} groups, {time.time() - t:.0f} s")
    return df

daily = scan("daily pass", """
    SELECT platform, app_day, week_start, period, is_election, automated_decision,
           n_filled_strictly_optional AS strict, n_filled_optional_fields AS opt, count(*) AS n
    FROM gold.sor GROUP BY ALL""")

dims = scan("dimension pass", """
    SELECT platform, category, automated_detection, automated_decision, content_type, count(*) AS n
    FROM gold.sor GROUP BY ALL""")

actions = scan("action pass", """
    SELECT platform, decision_visibility, decision_visibility_other, decision_provision,
           decision_account, count(*) AS n
    FROM gold.sor GROUP BY ALL""")

OPTIONAL = ["decision_visibility", "decision_visibility_other", "end_date_visibility_restriction",
            "decision_monetary", "decision_monetary_other", "end_date_monetary_restriction",
            "decision_provision", "end_date_service_restriction", "decision_account",
            "end_date_account_restriction", "account_type", "decision_ground_reference_url",
            "illegal_content_legal_ground", "incompatible_content_ground",
            "incompatible_content_illegal", "category_addition", "category_specification",
            "category_specification_other", "content_type_other", "content_language", "source_identity"]
# The two explanation attributes are excluded from the Gold view; their fill rates are read from
# Silver directly so that table 4 covers all 23 optional attributes.
fill = scan("fill-rate pass", "SELECT platform, count(*) AS n, " + ", ".join(
    f'count({c}) AS "{c}"' for c in OPTIONAL + ["illegal_content_explanation", "incompatible_content_explanation"]
) + ", sum(n_filled_strictly_optional) AS sum_strict, sum(n_filled_optional_fields) AS sum_opt "
    f"FROM silver.sor WHERE content_date < DATE '{SCHEMA_BREAK}' AND created_at < TIMESTAMP '{SCHEMA_BREAK}' GROUP BY platform")

totals = daily.groupby("platform")["n"].sum()
print("\nrows per platform:", totals.to_dict())
print("rows with application_date after 2025-06-30 (period = outside):",
      daily[daily.period == "outside"].groupby("platform")["n"].sum().to_dict())

daily pass: 3,799 groups, 114 s


dimension pass: 205 groups, 40 s


action pass: 79 groups, 131 s


fill-rate pass: 2 groups, 73 s

rows per platform: {'TikTok': 816446092, 'X': 132149}
rows with application_date after 2025-06-30 (period = outside): {}


### 2.2 Measure classification

Every free-text value of `decision_visibility_other` is assigned an action class and a measure type by
the rules below; appendix A lists the result value by value. A value naming several measures takes the
hardest. At statement level, the coded attributes and the free-text class are combined and the hardest
measure is kept, in the order account, monetary, provision of service, visibility (Groesch et al., 2026),
followed by advertisement not published, unspecified, no measure named and no measure reported. The
action class of a statement is the action of its hardest measure; within visibility, removal ranks above
disabling, age restriction, demotion and non-recommendation.

In [4]:
MEASURE_ORDER = ["account", "monetary", "provision of service", "visibility",
                 "ad not published", "unspecified", "no measure named"]


def tiktok_components(text):
    """All (measure type, action class) pairs named in one TikTok free-text value."""
    s = text.strip().lower()
    comps = []
    if any(k in s for k in ("settlement suspended", "funds deduction", "withholding of commissions")):
        comps.append(("monetary", "payment restriction"))
    if any(k in s for k in ("shop has been permanently closed", "shop will be closed", "shop closed",
                            "order volume", "campaign ban", "mega campaigns", "affiliate")):
        comps.append(("provision of service", "shop restriction"))
    if "generate sales" in s:
        comps.append(("provision of service", "creator sales restriction"))
    if any(k in s for k in ("deactivate products", "freeze products", "suspend products",
                            "removal of all violating products", "removal of products")):
        comps.append(("provision of service", "product restriction"))
    if "live access suspension" in s or s == "live feature":
        comps.append(("provision of service", "LIVE feature restriction"))
    if "not eligible for recommendation" in s:
        comps.append(("visibility", "not recommended in feed and search" if "search" in s
                      else "not recommended in feed"))
    if "product review removal" in s or "product review visible to self" in s:
        comps.append(("visibility", "review removed or hidden"))
    if "ad is not eligible" in s:
        comps.append(("ad not published", "ad not published"))
    if "violation point" in s or s == "warning":
        comps.append(("no measure named", "violation points or warning"))
    if s in ("cl3", "cl4", "restriction"):
        comps.append(("unspecified", "unspecified code or term"))
    return comps


X_CLASSES = {
    "Bounce": ("provision of service", "service partially suspended"),
    "Suspend": ("account", "account suspended"),
    "Deactivate": ("account", "account terminated"),
    "RemoveVerifiedType": ("provision of service", "verification removed"),
    "RemoveBlueVerification": ("provision of service", "verification removed"),
    "AddTweetAnnotation": ("visibility", "annotation added"),
    "Nsfw": ("no measure named", "NSFW, no measure named"),
    "Unban": ("no measure named", "reversal (unban)"),
    "Ban": ("unspecified", "ban, type not stated"),
}


def classify(platform, text):
    comps = [X_CLASSES[text]] if platform == "X" else tiktok_components(text)
    assert comps, f"unclassified free-text value: {platform}: {text!r}"
    return min(comps, key=lambda c: MEASURE_ORDER.index(c[0]))


classes = (actions.dropna(subset=["decision_visibility_other"])
             .groupby(["platform", "decision_visibility_other"], as_index=False)["n"].sum())
classes[["measure_type", "action_class"]] = [classify(p, t) for p, t in
                                             zip(classes.platform, classes.decision_visibility_other)]
print(classes.groupby("platform").size().to_dict(), "distinct free-text values, all classified")

sql_str = lambda v: "'" + v.replace("'", "''") + "'"
con.execute("CREATE OR REPLACE VIEW gold.free_text_classes AS SELECT * FROM (VALUES " + ", ".join(
    f"({sql_str(r.platform)}, {sql_str(r.decision_visibility_other)}, {sql_str(r.measure_type)}, "
    f"{sql_str(r.action_class)})" for r in classes.itertuples()
) + ") t(platform, free_text, measure_type, action_class)")

V = "decision_visibility"
con.execute(f"""
    CREATE OR REPLACE VIEW gold.measures AS
    SELECT s.*,
        CASE
            WHEN s.decision_account IS NOT NULL OR f.measure_type = 'account' THEN 'account'
            WHEN s.decision_monetary IS NOT NULL OR f.measure_type = 'monetary' THEN 'monetary'
            WHEN s.decision_provision IS NOT NULL OR f.measure_type = 'provision of service' THEN 'provision of service'
            WHEN list_has_any(s.{V}, ['DECISION_VISIBILITY_CONTENT_REMOVED', 'DECISION_VISIBILITY_CONTENT_DISABLED',
                                      'DECISION_VISIBILITY_CONTENT_AGE_RESTRICTED', 'DECISION_VISIBILITY_CONTENT_DEMOTED'])
                 OR f.measure_type = 'visibility' THEN 'visibility'
            WHEN f.measure_type IS NOT NULL THEN f.measure_type
            ELSE 'no measure reported'
        END AS measure_type,
        CASE
            WHEN s.decision_account = 'DECISION_ACCOUNT_TERMINATED' THEN 'account terminated'
            WHEN s.decision_account = 'DECISION_ACCOUNT_SUSPENDED' THEN 'account suspended'
            WHEN f.measure_type = 'account' THEN f.action_class
            WHEN s.decision_monetary IS NOT NULL THEN lower(replace(s.decision_monetary, 'DECISION_MONETARY_', 'monetary: '))
            WHEN f.measure_type = 'monetary' THEN f.action_class
            WHEN s.decision_provision = 'DECISION_PROVISION_TOTAL_TERMINATION' THEN 'service terminated'
            WHEN s.decision_provision = 'DECISION_PROVISION_TOTAL_SUSPENSION' THEN 'service suspended'
            WHEN s.decision_provision = 'DECISION_PROVISION_PARTIAL_SUSPENSION' THEN 'service partially suspended'
            WHEN f.measure_type = 'provision of service' THEN f.action_class
            WHEN list_contains(s.{V}, 'DECISION_VISIBILITY_CONTENT_REMOVED') THEN 'content removed'
            WHEN list_contains(s.{V}, 'DECISION_VISIBILITY_CONTENT_DISABLED') THEN 'content disabled'
            WHEN list_contains(s.{V}, 'DECISION_VISIBILITY_CONTENT_AGE_RESTRICTED') THEN 'age restricted'
            WHEN list_contains(s.{V}, 'DECISION_VISIBILITY_CONTENT_DEMOTED') THEN 'demoted'
            WHEN f.measure_type IS NOT NULL THEN f.action_class
            ELSE 'no measure reported'
        END AS action_class
    FROM gold.sor s
    LEFT JOIN gold.free_text_classes f
        ON s.platform = f.platform AND s.decision_visibility_other = f.free_text
""")

measures = scan("measure pass", """
    SELECT platform, period, is_election, content_type, measure_type, action_class, count(*) AS n
    FROM gold.measures GROUP BY ALL""")
assert (measures.groupby("platform")["n"].sum() == totals).all(), "measure pass does not reconcile"

{'TikTok': 61, 'X': 9} distinct free-text values, all classified


measure pass: 275 groups, 163 s


## 3. RQ1: platform comparison and modality

Population: the full analysis set of each platform, without period restriction. Every share is shown
next to its denominator.

### Table 1a: coded `decision_visibility`

A statement can carry several values; each statement is counted once per value, so counts within a
platform can sum to more than the platform total. `(none)` marks statements without a visibility
value.

In [5]:
vis = actions[["platform", "decision_visibility", "n"]].copy()
vis["value"] = vis["decision_visibility"].map(lambda v: list(v) if hasattr(v, "__len__") else ["(none)"])
vis = vis.explode("value").groupby(["platform", "value"], as_index=False)["n"].sum()
vis["platform_total"] = vis["platform"].map(totals)
vis["share_of_statements"] = (vis["n"] / vis["platform_total"]).round(6)
freeze(vis.sort_values(["platform", "n"], ascending=[True, False]), "t01a_decision_visibility_coded")

,platform,value,n,platform_total,share_of_statements
4,TikTok,DECISION_VISIBILITY_CONTENT_REMOVED,499851604,816446092,0.612229
5,TikTok,DECISION_VISIBILITY_OTHER,306937159,816446092,0.375943
0,TikTok,(none),6427889,816446092,0.007873
1,TikTok,DECISION_VISIBILITY_CONTENT_AGE_RESTRICTED,3418480,816446092,0.004187
2,TikTok,DECISION_VISIBILITY_CONTENT_DEMOTED,27928,816446092,0.000034
3,TikTok,DECISION_VISIBILITY_CONTENT_DISABLED,19760,816446092,0.000024
6,X,DECISION_VISIBILITY_OTHER,132149,132149,1.000000


On TikTok, 61.22% of the 816,446,092 statements carry the coded value `CONTENT_REMOVED` and 37.59% the value `OTHER`; 0.79% carry no visibility value. X reports `OTHER` for all 132,149 statements.

### Table 1b: coded service and account measures

In [6]:
rows = []
for attr in ("decision_provision", "decision_account"):
    d = actions.groupby(["platform", attr], dropna=False, as_index=False)["n"].sum()
    d = d.rename(columns={attr: "value"}).assign(attribute=attr)
    d["value"] = d["value"].fillna("(none)")
    rows.append(d)
t1b = pd.concat(rows)[["platform", "attribute", "value", "n"]]
t1b["platform_total"] = t1b["platform"].map(totals)
t1b["share_of_statements"] = (t1b["n"] / t1b["platform_total"]).round(6)
freeze(t1b.sort_values(["platform", "attribute", "n"], ascending=[True, True, False]), "t01b_provision_account_coded")

,platform,attribute,value,n,platform_total,share_of_statements
2,TikTok,decision_account,(none),812956350,816446092,0.995726
1,TikTok,decision_account,DECISION_ACCOUNT_TERMINATED,3488446,816446092,0.004273
0,TikTok,decision_account,DECISION_ACCOUNT_SUSPENDED,1296,816446092,0.000002
1,TikTok,decision_provision,(none),813507945,816446092,0.996401
0,TikTok,decision_provision,DECISION_PROVISION_PARTIAL_SUSPENSION,2938147,816446092,0.003599
5,X,decision_account,(none),110032,132149,0.832636
3,X,decision_account,DECISION_ACCOUNT_SUSPENDED,21998,132149,0.166464
4,X,decision_account,DECISION_ACCOUNT_TERMINATED,119,132149,0.000900
2,X,decision_provision,DECISION_PROVISION_PARTIAL_SUSPENSION,75031,132149,0.567776
5,X,decision_provision,(none),35001,132149,0.264860


X reports a partial suspension of service for 56.78% of its 132,149 statements, a total suspension for 16.65% and a termination for 0.09%; the account attribute records the same 21,998 suspensions and 119 terminations. On TikTok, account terminations make up 0.43% and partial service suspensions 0.36% of 816,446,092 statements.

### Table 1d: measure types after classification

One measure type per statement: the hardest measure among the coded attributes and the classified free
text (section 2.2). The second table splits each measure type into action classes.

In [7]:
TYPE_ORDER = MEASURE_ORDER + ["no measure reported"]
t1d = measures.groupby(["platform", "measure_type"], as_index=False)["n"].sum()
t1d["platform_total"] = t1d["platform"].map(totals)
t1d["share_of_statements"] = (t1d["n"] / t1d["platform_total"]).round(6)
t1d["measure_type"] = pd.Categorical(t1d["measure_type"], TYPE_ORDER, ordered=True)
freeze(t1d.sort_values(["platform", "measure_type"]), "t01d_measure_types")

,platform,measure_type,n,platform_total,share_of_statements
0,TikTok,account,3489742,816446092,0.004274
2,TikTok,monetary,45,816446092,0.000000
4,TikTok,provision of service,4101931,816446092,0.005024
6,TikTok,visibility,807858377,816446092,0.989482
1,TikTok,ad not published,759507,816446092,0.000930
5,TikTok,unspecified,236105,816446092,0.000289
3,TikTok,no measure named,385,816446092,0.000000
7,X,account,22117,132149,0.167364
9,X,provision of service,75034,132149,0.567798
11,X,visibility,6,132149,0.000045


After classification, visibility is the hardest measure for 98.95% of TikTok's 816,446,092 statements, followed by provision of service (0.50%) and account measures (0.43%); 236,105 statements (0.03%) carry only an unspecified measure. On X, provision of service is the hardest measure for 56.78% of 132,149 statements and account measures for 16.74%, while 26.45% name no measure; visibility measures occur in 6 statements.

In [8]:
t1d2 = measures.groupby(["platform", "measure_type", "action_class"], as_index=False)["n"].sum()
t1d2["platform_total"] = t1d2["platform"].map(totals)
t1d2["share_of_statements"] = (t1d2["n"] / t1d2["platform_total"]).round(6)
t1d2["measure_type"] = pd.Categorical(t1d2["measure_type"], TYPE_ORDER, ordered=True)
freeze(t1d2.sort_values(["platform", "measure_type", "n"], ascending=[True, True, False]),
       "t01d2_action_classes")

,platform,measure_type,action_class,n,platform_total,share_of_statements
1,TikTok,account,account terminated,3488446,816446092,0.004273
0,TikTok,account,account suspended,1296,816446092,0.000002
4,TikTok,monetary,payment restriction,34,816446092,0.000000
3,TikTok,monetary,monetary: suspension,11,816446092,0.000000
9,TikTok,provision of service,service partially suspended,2938136,816446092,0.003599
8,TikTok,provision of service,product restriction,702800,816446092,0.000861
10,TikTok,provision of service,shop restriction,324597,816446092,0.000398
6,TikTok,provision of service,LIVE feature restriction,136080,816446092,0.000167
7,TikTok,provision of service,creator sales restriction,318,816446092,0.000000
14,TikTok,visibility,content removed,499851604,816446092,0.612229


Among TikTok's 816,446,092 statements, 499,851,604 (61.22%) report removal and 304,715,040 (37.32%) non-recommendation, of which 23,754,155 also restrict visibility in search. On X, 34,944 of 132,149 statements (26.44%) carry the label "Nsfw" without any restriction being named.

### Table 2: automation, detection × decision

In [9]:
t2 = dims.groupby(["platform", "automated_detection", "automated_decision"], as_index=False)["n"].sum()
t2["platform_total"] = t2["platform"].map(totals)
t2["share"] = (t2["n"] / t2["platform_total"]).round(6)
freeze(t2.sort_values(["platform", "n"], ascending=[True, False]), "t02_automation_crosstab")

,platform,automated_detection,automated_decision,n,platform_total,share
3,TikTok,Yes,AUTOMATED_DECISION_FULLY,782904141,816446092,0.958917
4,TikTok,Yes,AUTOMATED_DECISION_NOT_AUTOMATED,25733570,816446092,0.031519
1,TikTok,No,AUTOMATED_DECISION_NOT_AUTOMATED,6059756,816446092,0.007422
0,TikTok,No,AUTOMATED_DECISION_FULLY,1748622,816446092,0.002142
2,TikTok,No,AUTOMATED_DECISION_PARTIALLY,3,816446092,0.000000
6,X,No,AUTOMATED_DECISION_NOT_AUTOMATED,132143,132149,0.999955
5,X,No,AUTOMATED_DECISION_FULLY,6,132149,0.000045


On TikTok, 95.89% of 816,446,092 statements are detected and decided automatically, 3.15% are detected automatically and decided manually, and 0.74% are detected and decided manually. On X, 132,143 of 132,149 statements (99.995%) are detected and decided manually; 6 are decided automatically.

### Table 3: automation by `content_type`, TikTok

Each statement is counted once per content type it lists. `n_rows` is the denominator of the shares.

In [10]:
ct = dims[dims.platform == "TikTok"].copy()
ct["content_type"] = ct["content_type"].map(lambda v: list(v) if hasattr(v, "__len__") else ["(none)"])
ct = ct.explode("content_type")
t3 = ct.pivot_table(index="content_type", columns="automated_decision", values="n",
                    aggfunc="sum", fill_value=0)
t3.insert(0, "n_rows", t3.sum(axis=1))
for c in [c for c in t3.columns if c.startswith("AUTOMATED_DECISION_")]:
    t3[f"share_{c.removeprefix('AUTOMATED_DECISION_').lower()}"] = (t3[c] / t3["n_rows"]).round(6)
t3 = t3.sort_values("n_rows", ascending=False).reset_index()
freeze(t3, "t03_tiktok_automation_by_content_type")

automated_decision,content_type,n_rows,AUTOMATED_DECISION_FULLY,AUTOMATED_DECISION_NOT_AUTOMATED,AUTOMATED_DECISION_PARTIALLY,share_fully,share_not_automated,share_partially
0,CONTENT_TYPE_TEXT,452417893,450053018,2364875,0,0.994773,0.005227,0.0
1,CONTENT_TYPE_VIDEO,280946700,262047512,18899185,3,0.932730,0.067270,0.0
2,CONTENT_TYPE_IMAGE,65973668,59122667,6850998,3,0.896156,0.103844,0.0
3,CONTENT_TYPE_OTHER,16393626,13159690,3233936,0,0.802732,0.197268,0.0
4,CONTENT_TYPE_AUDIO,734375,269871,464504,0,0.367484,0.632516,0.0
5,CONTENT_TYPE_PRODUCT,36,5,31,0,0.138889,0.861111,0.0


The share of fully automated decisions on TikTok is 99.48% for text (452,417,893 statements), 93.27% for video (280,946,700), 89.62% for image (65,973,668), 80.27% for other content types (16,393,626) and 36.75% for audio (734,375).

### Table 3b: content types per platform

Content-type distribution of the analysis set for both platforms. As in table 3, a statement is counted
once per content type it lists, so the shares of a platform can sum to slightly more than 100%. The
denominator is the platform's number of statements. TikTok's counts are taken from table 3; X's come from
one query on its partitions.

In [11]:
t3_frozen = pd.read_csv(GOLD_OUT / "t03_tiktok_automation_by_content_type.csv")
platform_total = pd.read_csv(GOLD_OUT / "t01d_measure_types.csv").groupby("platform")["platform_total"].first()

x_types = con.sql("""
    SELECT content_type, count(*) AS n FROM gold.sor WHERE platform = 'X' GROUP BY content_type""").df()
x_types["content_type"] = x_types["content_type"].map(lambda v: list(v) if hasattr(v, "__len__") else ["(none)"])
x_types = x_types.explode("content_type").groupby("content_type", as_index=False)["n"].sum()

t3b = pd.concat([
    t3_frozen[["content_type", "n_rows"]].rename(columns={"n_rows": "n"}).assign(platform="TikTok"),
    x_types.assign(platform="X"),
])[["platform", "content_type", "n"]]
t3b["platform_total"] = t3b["platform"].map(platform_total)
t3b["share_of_statements"] = (t3b["n"] / t3b["platform_total"]).round(6)
freeze(t3b.sort_values(["platform", "n"], ascending=[True, False]), "t03b_content_types_by_platform")

,platform,content_type,n,platform_total,share_of_statements
0,TikTok,CONTENT_TYPE_TEXT,452417893,816446092,0.554131
1,TikTok,CONTENT_TYPE_VIDEO,280946700,816446092,0.344109
2,TikTok,CONTENT_TYPE_IMAGE,65973668,816446092,0.080806
3,TikTok,CONTENT_TYPE_OTHER,16393626,816446092,0.020079
4,TikTok,CONTENT_TYPE_AUDIO,734375,816446092,0.000899
5,TikTok,CONTENT_TYPE_PRODUCT,36,816446092,0.000000
1,X,CONTENT_TYPE_SYNTHETIC_MEDIA,132142,132149,0.999947
0,X,CONTENT_TYPE_AUDIO,7,132149,0.000053


Of TikTok's 816,446,092 statements, 55.41% concern text, 34.41% video, 8.08% image, 2.01% other content types, 0.09% audio and 0.000004% products. X classifies 132,142 of its 132,149 statements (99.995%) as synthetic media and 7 as audio; it reports no text, video or image content.

### Table 4: granularity

Mean of both granularity counts per platform, and the fill rate of each of the 23 optional attributes.
`strictly_optional` marks the 16 attributes counted by `n_filled_strictly_optional`.

In [12]:
STRICT_EXCLUDED = {"decision_visibility_other", "decision_monetary_other", "illegal_content_legal_ground",
                   "illegal_content_explanation", "incompatible_content_ground",
                   "incompatible_content_explanation", "content_type_other"}
f = fill.set_index("platform")
t4_means = pd.DataFrame({
    "n_rows": f["n"],
    "mean_n_filled_strictly_optional": (f["sum_strict"] / f["n"]).round(4),
    "mean_n_filled_optional_fields": (f["sum_opt"] / f["n"]).round(4),
}).reset_index()
freeze(t4_means, "t04a_granularity_means")
print(t4_means.to_string(index=False))

attrs = OPTIONAL + ["illegal_content_explanation", "incompatible_content_explanation"]
t4 = pd.DataFrame({"attribute": attrs,
                   "strictly_optional": [a not in STRICT_EXCLUDED for a in attrs]})
for plat in f.index:
    t4[f"{plat}_filled"] = [int(f.loc[plat, a]) for a in attrs]
    t4[f"{plat}_fill_rate"] = (t4[f"{plat}_filled"] / f.loc[plat, "n"]).round(6)
t4["rows_tiktok"], t4["rows_x"] = int(f.loc["TikTok", "n"]), int(f.loc["X", "n"])
freeze(t4.sort_values(["strictly_optional", "attribute"], ascending=[False, True]), "t04b_fill_rates")

platform    n_rows  mean_n_filled_strictly_optional  mean_n_filled_optional_fields
       X    132149                           3.3608                         6.3608
  TikTok 816446092                           1.0080                         3.4041


,attribute,strictly_optional,X_filled,X_fill_rate,TikTok_filled,TikTok_fill_rate,rows_tiktok,rows_x
10,account_type,True,0,0.000000,0,0.000000,816446092,132149
15,category_addition,True,33,0.000250,0,0.000000,816446092,132149
16,category_specification,True,132149,1.000000,0,0.000000,816446092,132149
17,category_specification_other,True,60528,0.458028,0,0.000000,816446092,132149
19,content_language,True,0,0.000000,0,0.000000,816446092,132149
8,decision_account,True,22117,0.167364,3489742,0.004274,816446092,132149
11,decision_ground_reference_url,True,0,0.000000,0,0.000000,816446092,132149
3,decision_monetary,True,0,0.000000,11,0.000000,816446092,132149
6,decision_provision,True,97148,0.735140,2938147,0.003599,816446092,132149
0,decision_visibility,True,132149,1.000000,810018203,0.992127,816446092,132149


X fills on average 3.36 of the 16 strictly optional attributes (132,149 statements) and TikTok 1.01 (816,446,092 statements); across all 23 optional attributes the means are 6.36 and 3.40. X fills `illegal_content_legal_ground` and `illegal_content_explanation` in every statement and never the `incompatible_content_*` attributes, while TikTok fills `incompatible_content_ground` and `incompatible_content_explanation` in 99.998% of statements.

### Table 4c: `decision_facts` per platform

Number of distinct values of `decision_facts` per platform, also after normalisation (surrounding
whitespace removed, internal whitespace collapsed, trailing full stops removed), and the ten most frequent
values with count and share. The denominator is the platform's number of statements in the analysis set.
`decision_facts` is excluded from the Gold views, so this table alone reads it from `silver.analysis_*`.

In [13]:
def normalise(text):
    return " ".join(str(text).split()).rstrip(".").rstrip()


rows = []
for view, name in (("silver.analysis_tiktok", "TikTok"), ("silver.analysis_x", "X")):
    facts = con.sql(f"SELECT decision_facts, count(*) AS n FROM {view} GROUP BY 1").df()
    total = int(facts["n"].sum())
    n_distinct = facts["decision_facts"].nunique(dropna=False)
    n_distinct_norm = facts["decision_facts"].map(normalise).nunique(dropna=False)
    top = facts.sort_values(["n", "decision_facts"], ascending=[False, True]).head(10).reset_index(drop=True)
    for rank, r in top.iterrows():
        rows.append({"platform": name, "rank": rank + 1, "decision_facts": r["decision_facts"], "n": int(r["n"]),
                     "platform_total": total, "share_of_statements": round(r["n"] / total, 6),
                     "n_distinct": n_distinct, "n_distinct_normalised": n_distinct_norm})
    print(f"{name}: {total:,} statements, {n_distinct} distinct values, {n_distinct_norm} after normalisation")

t4c = pd.DataFrame(rows)
freeze(t4c, "t04c_decision_facts")

TikTok: 816,446,092 statements, 8 distinct values, 7 after normalisation
X: 132,149 statements, 10 distinct values, 10 after normalisation


,platform,rank,decision_facts,n,platform_total,share_of_statements,n_distinct,n_distinct_normalised
0,TikTok,1,The decision was taken pursuant to own-initiative investigations.,800668501,816446092,0.980675,8,7
1,TikTok,2,The decision was taken pursuant to own-initiative investigations,10675390,816446092,0.013075,8,7
2,TikTok,3,The decision was taken in response to a report alleging a violation of our Community G...,5014893,816446092,0.006142,8,7
3,TikTok,4,The decision was taken in response to a report alleging a violation of our Advertising...,66520,816446092,0.000081,8,7
4,TikTok,5,The decision was taken in response to an illegal content report under Art. 16 DSA.,20142,816446092,0.000025,8,7
5,TikTok,6,We have taken this decision based on a report under the ElDigital Services Act.,402,816446092,0.000000,8,7
6,TikTok,7,The decision was taken in response to a report from a third party.,233,816446092,0.000000,8,7
7,TikTok,8,The decision was taken in response to a report from a Trusted Flagger under Art. 22 DSA.,11,816446092,0.000000,8,7
8,X,1,Decision - action Bounce triggered manually by an agent,75025,132149,0.567730,10,10
9,X,2,Decision - action Nsfw triggered manually by an agent,34944,132149,0.264429,10,10


TikTok uses 8 distinct values of `decision_facts` across 816,446,092 statements, 7 after normalisation. "The decision was taken pursuant to own-initiative investigations." accounts for 98.07%, and the same sentence without the final full stop for a further 1.31%; each wording referring to a report accounts for 0.61% or less. X uses 10 distinct values across 132,149 statements, each naming an internal action (Bounce 56.77%, Nsfw 26.44%, Suspend 16.65%); 132,143 state that the action was triggered manually by an agent and 6 that it was triggered by the system with no human intervention.

## 4. RQ2: election period, within each platform

Population: statements with `application_date` from 2025-01-01 to 2025-06-30. X has no statement in the
election category; its election columns are zero by measurement, not missing.

In [14]:
w = daily[daily.period != "outside"].copy()
w["n_el"] = w["n"].where(w["is_election"], 0)
w["n_fully"] = w["n"].where(w["automated_decision"] == "AUTOMATED_DECISION_FULLY", 0)
w["sum_strict"] = w["strict"] * w["n"]
w["sum_opt"] = w["opt"] * w["n"]
PERIOD_NAMES = [p for p, _, _ in PERIODS]


def period_label(periods):
    p = sorted(set(periods), key=PERIOD_NAMES.index)
    return p[0] if len(p) == 1 else "mixed: " + "/".join(p)


weekly = (w.groupby(["platform", "week_start"])
            .agg(days_in_window=("app_day", "nunique"), period_label=("period", period_label),
                 n_total=("n", "sum"), n_election=("n_el", "sum"), n_fully_automated=("n_fully", "sum"),
                 sum_strict=("sum_strict", "sum"), sum_opt=("sum_opt", "sum"))
            .reset_index())

### Table 5: weekly election share

In [15]:
t5 = weekly[["platform", "week_start", "days_in_window", "period_label", "n_total", "n_election"]].copy()
t5["share_election"] = (t5["n_election"] / t5["n_total"]).round(6)
freeze(t5, "t05_weekly_election_share")

,platform,week_start,days_in_window,period_label,n_total,n_election,share_election
0,TikTok,2024-12-30,5,baseline_a,14777457,423872,0.028684
1,TikTok,2025-01-06,7,baseline_a,19488042,672337,0.034500
2,TikTok,2025-01-13,7,baseline_a,19530378,563361,0.028845
3,TikTok,2025-01-20,7,campaign,29322013,1781460,0.060755
4,TikTok,2025-01-27,7,campaign,32886824,2252477,0.068492
5,TikTok,2025-02-03,7,campaign,32328730,1680000,0.051966
6,TikTok,2025-02-10,7,campaign,36813273,2368521,0.064339
7,TikTok,2025-02-17,7,campaign,42065282,2539242,0.060364
8,TikTok,2025-02-24,7,post_election,44806599,2678578,0.059781
9,TikTok,2025-03-03,7,post_election,43784858,2621488,0.059872


The weekly election-category share on TikTok ranges from 2.87% to 3.45% in the three weeks of the first baseline, from 5.20% to 6.85% in the five campaign weeks and from 2.27% to 4.10% in the weeks of the second baseline. X reports no election-category statement in any week.

### Table 6: period summary

Computed from days, so every statement falls into exactly one period. `rel_to_baseline_a` divides each
period's share by the first baseline's share of the same platform; the second baseline appears as its
own row, so the two baselines can be compared directly.

In [16]:
t6 = (w.groupby(["platform", "period"])
        .agg(days=("app_day", "nunique"), n_total=("n", "sum"), n_election=("n_el", "sum"))
        .reset_index())
t6["period"] = pd.Categorical(t6["period"], PERIOD_NAMES, ordered=True)
t6 = t6.sort_values(["platform", "period"])
t6["share_election"] = (t6["n_election"] / t6["n_total"]).round(6)
base_a = t6[t6.period == "baseline_a"].set_index("platform")["share_election"]
t6["rel_to_baseline_a"] = (t6["share_election"] / t6["platform"].map(base_a)).round(4)
freeze(t6, "t06_period_summary")

,platform,period,days,n_total,n_election,share_election,rel_to_baseline_a
0,TikTok,baseline_a,19,53795877,1659570,0.030849,1.0000
2,TikTok,campaign,35,173416122,10621700,0.061250,1.9855
3,TikTok,post_election,36,230517382,13232573,0.057404,1.8608
4,TikTok,transition,30,183024086,6500766,0.035519,1.1514
1,TikTok,baseline_b,61,175692625,5212782,0.029670,0.9618
5,X,baseline_a,19,25659,0,0.000000,NaN
7,X,campaign,35,38503,0,0.000000,NaN
8,X,post_election,36,36631,0,0.000000,NaN
9,X,transition,30,13586,0,0.000000,NaN
6,X,baseline_b,61,17770,0,0.000000,NaN


The election category accounts for 3.085% of TikTok statements in the first baseline period (1,659,570 of 53,795,877), 6.125% during the campaign (10,621,700 of 173,416,122), 5.740% after the election (13,232,573 of 230,517,382), 3.552% in the transition (6,500,766 of 183,024,086) and 2.967% in the second baseline period (5,212,782 of 175,692,625). Relative to the first baseline, the campaign share is 1.99 times as high and the second-baseline share 0.96 times.

### Table 7: weekly automation share

In [17]:
t7 = weekly[["platform", "week_start", "days_in_window", "period_label", "n_total", "n_fully_automated"]].copy()
t7["share_fully_automated"] = (t7["n_fully_automated"] / t7["n_total"]).round(6)
freeze(t7, "t07_weekly_automation_share")

,platform,week_start,days_in_window,period_label,n_total,n_fully_automated,share_fully_automated
0,TikTok,2024-12-30,5,baseline_a,14777457,14295327,0.967374
1,TikTok,2025-01-06,7,baseline_a,19488042,18740070,0.961619
2,TikTok,2025-01-13,7,baseline_a,19530378,18779563,0.961557
3,TikTok,2025-01-20,7,campaign,29322013,28148660,0.959984
4,TikTok,2025-01-27,7,campaign,32886824,31437926,0.955943
5,TikTok,2025-02-03,7,campaign,32328730,30828552,0.953596
6,TikTok,2025-02-10,7,campaign,36813273,35382857,0.961144
7,TikTok,2025-02-17,7,campaign,42065282,40406686,0.960571
8,TikTok,2025-02-24,7,post_election,44806599,43410906,0.968851
9,TikTok,2025-03-03,7,post_election,43784858,42340837,0.967020


TikTok's weekly share of fully automated decisions lies between 93.17% and 97.23% in every full week from January to June 2025; the one-day week of 30 June shows 92.03%. On X, the share is zero in all but six weeks, with a maximum of 0.06% (1 of 1,703 statements).

### Table 8: weekly mean granularity

In [18]:
t8 = weekly[["platform", "week_start", "days_in_window", "period_label", "n_total"]].copy()
t8["mean_n_filled_strictly_optional"] = (weekly["sum_strict"] / weekly["n_total"]).round(4)
t8["mean_n_filled_optional_fields"] = (weekly["sum_opt"] / weekly["n_total"]).round(4)
freeze(t8, "t08_weekly_granularity")

,platform,week_start,days_in_window,period_label,n_total,mean_n_filled_strictly_optional,mean_n_filled_optional_fields
0,TikTok,2024-12-30,5,baseline_a,14777457,1.0130,3.4243
1,TikTok,2025-01-06,7,baseline_a,19488042,1.0135,3.4366
2,TikTok,2025-01-13,7,baseline_a,19530378,1.0133,3.4584
3,TikTok,2025-01-20,7,campaign,29322013,1.0111,3.4598
4,TikTok,2025-01-27,7,campaign,32886824,1.0092,3.4350
5,TikTok,2025-02-03,7,campaign,32328730,1.0094,3.4328
6,TikTok,2025-02-10,7,campaign,36813273,1.0087,3.4288
7,TikTok,2025-02-17,7,campaign,42065282,1.0085,3.4217
8,TikTok,2025-02-24,7,post_election,44806599,1.0074,3.3929
9,TikTok,2025-03-03,7,post_election,43784858,1.0082,3.4168


Across all weeks from January to June 2025, the weekly mean of the strict granularity count lies between 1.0003 and 1.0206 on TikTok and between 2.90 and 3.78 on X.

### Table 9: TikTok daily series, 2025-01-01 to 2025-03-31 (supplementary)

In [19]:
d = w[(w.platform == "TikTok") & (w.app_day <= pd.Timestamp("2025-03-31"))]
t9 = (d.groupby("app_day")
        .agg(period=("period", "first"), n_total=("n", "sum"), n_election=("n_el", "sum"),
             n_fully_automated=("n_fully", "sum"), sum_strict=("sum_strict", "sum"))
        .reset_index().rename(columns={"app_day": "date"}))
t9["share_election"] = (t9["n_election"] / t9["n_total"]).round(6)
t9["share_fully_automated"] = (t9["n_fully_automated"] / t9["n_total"]).round(6)
t9["mean_n_filled_strictly_optional"] = (t9.pop("sum_strict") / t9["n_total"]).round(4)
freeze(t9, "t09_tiktok_daily_jan_mar")

,date,period,n_total,n_election,n_fully_automated,share_election,share_fully_automated,mean_n_filled_strictly_optional
0,2025-01-01,baseline_a,2902217,71901,2828528,0.024775,0.974609,1.0120
1,2025-01-02,baseline_a,2876732,82933,2782608,0.028829,0.967281,1.0134
2,2025-01-03,baseline_a,2935527,87679,2832756,0.029868,0.964991,1.0136
3,2025-01-04,baseline_a,3021940,90849,2919920,0.030063,0.966240,1.0132
4,2025-01-05,baseline_a,3041041,90510,2931515,0.029763,0.963984,1.0127
5,2025-01-06,baseline_a,2868571,94193,2750284,0.032836,0.958764,1.0133
6,2025-01-07,baseline_a,2572831,90935,2458845,0.035344,0.955696,1.0154
7,2025-01-08,baseline_a,2644784,92318,2521590,0.034906,0.953420,1.0154
8,2025-01-09,baseline_a,2536523,108978,2440925,0.042964,0.962311,1.0137
9,2025-01-10,baseline_a,2768762,103102,2673743,0.037238,0.965682,1.0127


The daily TikTok series covers the 90 days from 1 January to 31 March 2025, with between 2,527,193 (13 January) and 7,241,157 (23 March) statements per day. The daily election share ranges from 2.32% to 4.30% in the first baseline and from 4.19% to 7.87% during the campaign, with its maximum on 14 February.

## 5. Automation by category and granularity by automation status

### Table 10: automation by category

Population: the full analysis set of each platform. `n_rows` is the denominator of the shares.

In [20]:
t10 = dims.pivot_table(index=["platform", "category"], columns="automated_decision", values="n",
                       aggfunc="sum", fill_value=0)
t10.insert(0, "n_rows", t10.sum(axis=1))
for c in [c for c in t10.columns if c.startswith("AUTOMATED_DECISION_")]:
    t10[f"share_{c.removeprefix('AUTOMATED_DECISION_').lower()}"] = (t10[c] / t10["n_rows"]).round(6)
t10 = t10.reset_index().sort_values(["platform", "n_rows"], ascending=[True, False])
freeze(t10, "t10_automation_by_category")

automated_decision,platform,category,n_rows,AUTOMATED_DECISION_FULLY,AUTOMATED_DECISION_NOT_AUTOMATED,AUTOMATED_DECISION_PARTIALLY,share_fully,share_not_automated,share_partially
2,TikTok,STATEMENT_CATEGORY_ILLEGAL_OR_HARMFUL_SPEECH,398720354,394045083,4675271,0,0.988274,0.011726,0.0
10,TikTok,STATEMENT_CATEGORY_SCOPE_OF_PLATFORM_SERVICE,316507660,304484024,12023633,3,0.962012,0.037988,0.0
4,TikTok,STATEMENT_CATEGORY_NEGATIVE_EFFECTS_ON_CIVIC_DISCOURSE_OR_ELECTIONS,37227391,36915706,311685,0,0.991628,0.008372,0.0
13,TikTok,STATEMENT_CATEGORY_VIOLENCE,29697246,27480416,2216830,0,0.925352,0.074648,0.0
7,TikTok,STATEMENT_CATEGORY_PROTECTION_OF_MINORS,11866806,4977182,6889624,0,0.419421,0.580579,0.0
9,TikTok,STATEMENT_CATEGORY_SCAMS_AND_FRAUD,11238853,8789290,2449563,0,0.782045,0.217955,0.0
6,TikTok,STATEMENT_CATEGORY_PORNOGRAPHY_OR_SEXUALIZED_CONTENT,5667034,3342679,2324355,0,0.589846,0.410154,0.0
1,TikTok,STATEMENT_CATEGORY_DATA_PROTECTION_AND_PRIVACY_VIOLATIONS,3589638,3198023,391615,0,0.890904,0.109096,0.0
11,TikTok,STATEMENT_CATEGORY_SELF_HARM,946824,531813,415011,0,0.561681,0.438319,0.0
0,TikTok,STATEMENT_CATEGORY_ANIMAL_WELFARE,936225,845603,90622,0,0.903205,0.096795,0.0


Fully automated decisions account for 99.16% of TikTok's 37,227,391 election-category statements, 98.83% of 398,720,354 statements on illegal or harmful speech and 96.20% of 316,507,660 on the scope of the platform service, compared with 41.94% of 11,866,806 on the protection of minors, 58.98% of 5,667,034 on pornography or sexualised content and 56.17% of 946,824 on self-harm. On X, no category has an automated share above 0.05% (6 of 13,641 statements on the protection of minors).

### Table 11: granularity by automation status, TikTok

Statements decided fully automatically versus statements whose decision is reported as not automated,
over the whole analysis set and per period. Means plus quartiles of both granularity counts; the full
distribution is frozen separately.

In [21]:
tk = daily[daily.platform == "TikTok"]


def summarise(g, col):
    g = g.groupby(col, as_index=False)["n"].sum().sort_values(col)
    cum = g["n"].cumsum() / g["n"].sum()
    q = lambda p: int(g[col].iloc[(cum.values >= p).argmax()])  # exact quantile from the histogram
    return pd.Series({f"mean_{col}": round((g[col] * g["n"]).sum() / g["n"].sum(), 4),
                      f"p25_{col}": q(.25), f"p50_{col}": q(.5), f"p75_{col}": q(.75)})


out = []
for scope, g in [("all", tk)] + [(p, tk[tk.period == p]) for p in PERIOD_NAMES] + [("outside", tk[tk.period == "outside"])]:
    for status, gs in g.groupby("automated_decision"):
        out.append(pd.concat([pd.Series({"period": scope, "automated_decision": status, "n_rows": gs["n"].sum()}),
                              summarise(gs, "strict"), summarise(gs, "opt")]))
t11 = pd.DataFrame(out)
freeze(t11, "t11_tiktok_granularity_by_automation")

hist = (tk.groupby(["period", "automated_decision", "strict", "opt"], as_index=False)["n"].sum()
          .rename(columns={"strict": "n_filled_strictly_optional", "opt": "n_filled_optional_fields"}))
freeze(hist, "t11b_tiktok_granularity_distribution")
t11

,period,automated_decision,n_rows,mean_strict,p25_strict,p50_strict,p75_strict,mean_opt,p25_opt,p50_opt,p75_opt
0,all,AUTOMATED_DECISION_FULLY,784652763,1.0034,1.0,1.0,1.0,3.3994,3.0,3.0,4.0
1,all,AUTOMATED_DECISION_NOT_AUTOMATED,31793326,1.1229,1.0,1.0,1.0,3.5194,3.0,3.0,4.0
2,all,AUTOMATED_DECISION_PARTIALLY,3,1.0000,1.0,1.0,1.0,3.0000,3.0,3.0,3.0
3,baseline_a,AUTOMATED_DECISION_FULLY,51814960,1.0064,1.0,1.0,1.0,3.4406,3.0,3.0,4.0
4,baseline_a,AUTOMATED_DECISION_NOT_AUTOMATED,1980917,1.1948,1.0,1.0,1.0,3.4534,3.0,3.0,4.0
5,campaign,AUTOMATED_DECISION_FULLY,166204681,1.0047,1.0,1.0,1.0,3.4332,3.0,3.0,4.0
6,campaign,AUTOMATED_DECISION_NOT_AUTOMATED,7211441,1.1139,1.0,1.0,1.0,3.4582,3.0,3.0,4.0
7,post_election,AUTOMATED_DECISION_FULLY,222403266,1.0037,1.0,1.0,1.0,3.4001,3.0,3.0,4.0
8,post_election,AUTOMATED_DECISION_NOT_AUTOMATED,8114116,1.1250,1.0,1.0,1.0,3.5115,3.0,3.0,4.0
9,transition,AUTOMATED_DECISION_FULLY,175081459,1.0034,1.0,1.0,1.0,3.3725,3.0,3.0,4.0


TikTok statements decided fully automatically fill on average 1.0034 strictly optional attributes (784,652,763 statements), statements not decided automatically 1.1229 (31,793,326 statements). The difference appears in every period, from 1.0007 versus 1.0359 in the second baseline to 1.0064 versus 1.1948 in the first; all quartiles equal 1 in both groups.

## 6. Cross-checks within TikTok

### Table 12: election-category statements by content type and period

For each period and content type: (a) the content type's share of the election category, with the
number of election-category statements in the period as denominator; (b) the election share within the
content type, with the content type's statements in the period as denominator. A statement listing
several content types is counted once per type, so the shares in (a) can sum to slightly more than one.

In [22]:
m = measures[(measures.platform == "TikTok") & (measures.period != "outside")].copy()
el_period = m[m.is_election].groupby("period")["n"].sum()
m["content_type"] = m["content_type"].map(lambda v: list(v) if hasattr(v, "__len__") else ["(none)"])
mx = m.explode("content_type")
mx["n_el"] = mx["n"].where(mx["is_election"], 0)
t12 = mx.groupby(["period", "content_type"], as_index=False).agg(n_type=("n", "sum"), n_election=("n_el", "sum"))
t12["n_election_period"] = t12["period"].map(el_period)
t12["share_of_election_category"] = (t12["n_election"] / t12["n_election_period"]).round(6)
t12["election_share_within_type"] = (t12["n_election"] / t12["n_type"]).round(6)
t12["period"] = pd.Categorical(t12["period"], PERIOD_NAMES, ordered=True)
freeze(t12.sort_values(["period", "n_type"], ascending=[True, False]), "t12_election_by_content_type")

,period,content_type,n_type,n_election,n_election_period,share_of_election_category,election_share_within_type
3,baseline_a,CONTENT_TYPE_TEXT,27501511,2779,1659570,0.001675,0.000101
4,baseline_a,CONTENT_TYPE_VIDEO,21528772,63140,1659570,0.038046,0.002933
1,baseline_a,CONTENT_TYPE_IMAGE,3986534,1593548,1659570,0.960217,0.399733
2,baseline_a,CONTENT_TYPE_OTHER,731229,1,1659570,0.000001,0.000001
0,baseline_a,CONTENT_TYPE_AUDIO,49913,103,1659570,0.000062,0.002064
14,campaign,CONTENT_TYPE_TEXT,89749392,11082,10621700,0.001043,0.000123
15,campaign,CONTENT_TYPE_VIDEO,63666187,284123,10621700,0.026749,0.004463
12,campaign,CONTENT_TYPE_IMAGE,16963133,10326181,10621700,0.972178,0.608743
13,campaign,CONTENT_TYPE_OTHER,2890607,25,10621700,0.000002,0.000009
11,campaign,CONTENT_TYPE_AUDIO,152167,289,10621700,0.000027,0.001899


Image content makes up 96.02% of TikTok's election-category statements in the first baseline period, 97.22% during the campaign and 96.73% in the second baseline period (of 1,659,570, 10,621,700 and 5,212,782 statements). Within image content, the election share is 39.97% in the first baseline (3,986,534 image statements), 60.87% during the campaign (16,963,133) and 48.42% in the second baseline (10,413,315); for text and video it stays below 0.9% in every period.

### Table 13: measure types and visibility actions by period

(a) Measure-type distribution per period for all TikTok statements and for the election category only;
the denominator is the number of statements in the period and group. (b) Within the visibility type,
the action classes, separating removal from non-recommendation.

In [23]:
groups = [("all statements", m), ("election category", m[m.is_election])]
rows = []
for label, g in groups:
    d = g.groupby(["period", "measure_type"], as_index=False)["n"].sum()
    d["group"] = label
    rows.append(d)
t13a = pd.concat(rows)
t13a["period_total"] = t13a.groupby(["group", "period"])["n"].transform("sum")
t13a["share"] = (t13a["n"] / t13a["period_total"]).round(6)
t13a["period"] = pd.Categorical(t13a["period"], PERIOD_NAMES, ordered=True)
t13a["measure_type"] = pd.Categorical(t13a["measure_type"], TYPE_ORDER, ordered=True)
freeze(t13a.sort_values(["group", "period", "measure_type"])[["group", "period", "measure_type", "n",
                                                             "period_total", "share"]], "t13a_measure_types_by_period")

,group,period,measure_type,n,period_total,share
0,all statements,baseline_a,account,199258,53795877,0.003704
2,all statements,baseline_a,provision of service,380416,53795877,0.007071
4,all statements,baseline_a,visibility,53121729,53795877,0.987468
1,all statements,baseline_a,ad not published,47030,53795877,0.000874
3,all statements,baseline_a,unspecified,47444,53795877,0.000882
12,all statements,campaign,account,1112121,173416122,0.006413
14,all statements,campaign,provision of service,847821,173416122,0.004889
16,all statements,campaign,visibility,171288733,173416122,0.987732
13,all statements,campaign,ad not published,66262,173416122,0.000382
15,all statements,campaign,unspecified,101185,173416122,0.000583


Visibility is the hardest measure for between 98.75% and 99.19% of all TikTok statements in each period, and for at least 99.97% of election-category statements in each period.

In [24]:
rows = []
for label, g in groups:
    d = g[g.measure_type == "visibility"].groupby(["period", "action_class"], as_index=False)["n"].sum()
    d["group"] = label
    rows.append(d)
t13b = pd.concat(rows)
t13b["visibility_total"] = t13b.groupby(["group", "period"])["n"].transform("sum")
t13b["share_of_visibility"] = (t13b["n"] / t13b["visibility_total"]).round(6)
t13b["period"] = pd.Categorical(t13b["period"], PERIOD_NAMES, ordered=True)
freeze(t13b.sort_values(["group", "period", "n"], ascending=[True, True, False])[
    ["group", "period", "action_class", "n", "visibility_total", "share_of_visibility"]],
    "t13b_visibility_actions_by_period")

,group,period,action_class,n,visibility_total,share_of_visibility
2,all statements,baseline_a,content removed,30818967,53121729,0.580157
3,all statements,baseline_a,not recommended in feed,20402810,53121729,0.384077
4,all statements,baseline_a,not recommended in feed and search,1765098,53121729,0.033227
0,all statements,baseline_a,age restricted,134191,53121729,0.002526
1,all statements,baseline_a,content disabled,663,53121729,0.000012
12,all statements,campaign,content removed,100058705,171288733,0.584152
13,all statements,campaign,not recommended in feed,65258426,171288733,0.380985
14,all statements,campaign,not recommended in feed and search,5331346,171288733,0.031125
10,all statements,campaign,age restricted,635234,171288733,0.003709
11,all statements,campaign,content disabled,5022,171288733,0.000029


Among all TikTok visibility measures, removal accounts for 58.02% in the first baseline, 58.42% during the campaign and 65.20% in the second baseline, and non-recommendation (feed, or feed and search) for 41.73%, 41.21% and 34.19% (of 53,121,729, 171,288,733 and 173,574,959 statements). Within the election category, non-recommendation accounts for 89.37%, 90.82% and 82.16% of visibility measures and removal for 10.53%, 9.14% and 17.72% (of 1,659,453, 10,621,374 and 5,212,400 statements).

## Appendix A: classification of free-text measures

Every distinct value of `decision_visibility_other` in the analysis set, with its number of statements,
action class and measure type, as assigned by the rules in section 2.2.

In [25]:
t1c = classes.rename(columns={"decision_visibility_other": "free_text_value"})
t1c["platform_total"] = t1c["platform"].map(totals)
t1c["share_of_statements"] = (t1c["n"] / t1c["platform_total"]).round(6)
t1c["measure_type"] = pd.Categorical(t1c["measure_type"], TYPE_ORDER, ordered=True)
freeze(t1c.sort_values(["platform", "measure_type", "n"], ascending=[True, True, False]),
       "t01c_free_text_classification")

,platform,free_text_value,n,measure_type,action_class,platform_total,share_of_statements
56,TikTok,"permanent restriction of the account to generate sales from showcase, videos, and LIVE...",16,monetary,payment restriction,816446092,0.000000
58,TikTok,"permanent restriction of the account to generate sales from showcase, videos, and LIVE...",12,monetary,payment restriction,816446092,0.000000
16,TikTok,"90-day withholding of commissions. For more info, check the Creator Performance Evalua...",4,monetary,payment restriction,816446092,0.000000
17,TikTok,"90-day withholding of commissions. For more info, check the Creator Performance Evalua...",2,monetary,payment restriction,816446092,0.000000
21,TikTok,Deactivate products,697639,provision of service,product restriction,816446092,0.000854
54,TikTok,Your shop has been permanently closed.,324509,provision of service,shop restriction,816446092,0.000397
42,TikTok,Live feature,135755,provision of service,LIVE feature restriction,816446092,0.000166
59,TikTok,removal of all violating products from this video,3568,provision of service,product restriction,816446092,0.000004
25,TikTok,Freeze products,1127,provision of service,product restriction,816446092,0.000001
55,TikTok,"permanent restriction of the account to generate sales from showcase, videos, and LIVE...",280,provision of service,creator sales restriction,816446092,0.000000


## Appendix B: X submission pause, 2025-03-25 to 2025-04-15

No X dump file dated 2025-03-25 to 2025-04-15 contains data. The query lists, for the X statements
decided on those days, the dump in which they were submitted.

In [26]:
tB = con.sql("""
    SELECT dump_date, count(*) AS n, min(app_day) AS first_decision_day, max(app_day) AS last_decision_day,
           min(communication_delay) AS min_communication_delay, max(communication_delay) AS max_communication_delay
    FROM gold.sor
    WHERE platform = 'X' AND app_day BETWEEN DATE '2025-03-25' AND DATE '2025-04-15'
    GROUP BY dump_date ORDER BY dump_date""").df()
n_days = con.sql("""
    SELECT count(DISTINCT app_day) FROM gold.sor
    WHERE platform = 'X' AND app_day BETWEEN DATE '2025-03-25' AND DATE '2025-04-15'""").fetchone()[0]
print(f"{tB['n'].sum():,} statements from {len(tB)} dump files; {n_days} of 22 decision days represented")
freeze(tB, "tB_x_submission_pause")

16,879 statements from 15 dump files; 22 of 22 decision days represented


,dump_date,n,first_decision_day,last_decision_day,min_communication_delay,max_communication_delay
0,2025-04-17,2074,2025-03-25,2025-03-26,22,23
1,2025-04-18,2713,2025-03-26,2025-03-28,21,23
2,2025-04-19,1333,2025-03-28,2025-03-29,21,22
3,2025-04-20,1796,2025-03-29,2025-03-31,20,22
4,2025-04-21,1116,2025-03-31,2025-04-01,20,21
5,2025-04-22,608,2025-04-01,2025-04-03,19,21
6,2025-04-23,1168,2025-04-03,2025-04-05,18,20
7,2025-04-24,682,2025-04-05,2025-04-06,18,19
8,2025-04-25,1014,2025-04-06,2025-04-07,18,19
9,2025-04-26,927,2025-04-07,2025-04-09,17,19


All 16,879 X statements decided between 2025-03-25 and 2025-04-15 were submitted in the dumps of 2025-04-17 to 2025-05-01, with communication delays of 15 to 23 days; each of the 22 decision days is represented. The absence of data in the dumps of those days concerns submission, not moderation.

## Robustness: Germany-only statements

Population: TikTok statements of the analysis set whose territorial scope names Germany alone
(`n_countries_in_scope = 1` and `territorial_scope` contains `DE`). The section repeats the period and
weekly view of section 4 for statements that target Germany specifically. It relies only on the setup
and view cells, so it can be run without the aggregation passes of section 2.1.

In [27]:
PERIOD_NAMES = [p for p, _, _ in PERIODS]  # as in section 4; derived here so this section runs on its own

t = time.time()
de = con.sql("""
    SELECT app_day, week_start, period, is_election, content_type, automated_decision, count(*) AS n
    FROM gold.sor
    WHERE platform = 'TikTok' AND n_countries_in_scope = 1 AND list_contains(territorial_scope, 'DE')
    GROUP BY ALL""").df()
de["n_el"] = de["n"].where(de["is_election"], 0)
de["n_fully"] = de["n"].where(de["automated_decision"] == "AUTOMATED_DECISION_FULLY", 0)
print(f"Germany-only pass: {len(de):,} groups, {de['n'].sum():,} statements, "
      f"{int(de['n_el'].sum()):,} in the category, {time.time() - t:.0f} s")
outside = de[de.period == "outside"]
print(f"application_date after 2025-06-30: {int(outside['n'].sum()):,} statements, "
      f"{int(outside['n_el'].sum()):,} in the category")
de_w = de[de.period != "outside"]
period_days = {p: (pd.Timestamp(b) - pd.Timestamp(a)).days + 1 for p, a, b in PERIODS}

Germany-only pass: 762 groups, 1,595,276 statements, 243 in the category, 131 s
application_date after 2025-06-30: 0 statements, 0 in the category


### Table 14a: period summary, Germany-only statements

Layout of table 6. `days` are calendar days of the period.

In [28]:
t14a = de_w.groupby("period").agg(n_total=("n", "sum"), n_election=("n_el", "sum")).reset_index()
t14a.insert(1, "days", t14a["period"].map(period_days))
t14a["period"] = pd.Categorical(t14a["period"], PERIOD_NAMES, ordered=True)
t14a = t14a.sort_values("period")
t14a["share_election"] = (t14a["n_election"] / t14a["n_total"]).round(6)
base = t14a.loc[t14a.period == "baseline_a", "share_election"].iloc[0]
t14a["rel_to_baseline_a"] = (t14a["share_election"] / base).round(4) if base else float("nan")
low = t14a.loc[t14a["n_election"] < 1000, "period"].astype(str).tolist()
print(f"periods with fewer than 1,000 category statements: {len(low)} of {len(t14a)} ({', '.join(low)})")
freeze(t14a, "t14a_de_only_period_summary")

periods with fewer than 1,000 category statements: 5 of 5 (baseline_a, campaign, post_election, transition, baseline_b)


,period,days,n_total,n_election,share_election,rel_to_baseline_a
0,baseline_a,19,61993,1,0.000016,1.0000
2,campaign,35,86532,0,0.000000,0.0000
3,post_election,36,79027,21,0.000266,16.6250
4,transition,30,443415,202,0.000456,28.5000
1,baseline_b,61,924309,19,0.000021,1.3125


Among Germany-only TikTok statements, every period has fewer than 1,000 statements in the civic-discourse and elections category: 1 of 61,993 in the first baseline (0.002%), 0 of 86,532 during the campaign, 21 of 79,027 after the election (0.027%), 202 of 443,415 in the transition (0.046%) and 19 of 924,309 in the second baseline (0.002%). The check is therefore inconclusive, not negative; the ratios to the first baseline rest on a single category statement.

### Table 14b: weekly series, Germany-only statements

Layout of table 5. `days_in_window` counts the calendar days of the ISO week inside 2025-01-01 to
2025-06-30, so weeks are comparable with table 5 even where a day has no statement in the subset.

In [29]:
WINDOW = (pd.Timestamp(PERIODS[0][1]), pd.Timestamp(PERIODS[-1][2]))


def day_period(d):
    return next(p for p, a, b in PERIODS if pd.Timestamp(a) <= d <= pd.Timestamp(b))


rows = []
for wk in pd.date_range(WINDOW[0] - pd.Timedelta(days=WINDOW[0].weekday()), WINDOW[1], freq="7D"):
    days = [d for d in pd.date_range(wk, periods=7) if WINDOW[0] <= d <= WINDOW[1]]
    periods = sorted({day_period(d) for d in days}, key=PERIOD_NAMES.index)
    sub = de_w[de_w["week_start"] == wk]
    rows.append({"week_start": wk.date(), "days_in_window": len(days),
                 "period_label": periods[0] if len(periods) == 1 else "mixed: " + "/".join(periods),
                 "n_total": int(sub["n"].sum()), "n_election": int(sub["n_el"].sum())})
t14b = pd.DataFrame(rows)
t14b["share_election"] = (t14b["n_election"] / t14b["n_total"]).round(6)
freeze(t14b, "t14b_de_only_weekly")

,week_start,days_in_window,period_label,n_total,n_election,share_election
0,2024-12-30,5,baseline_a,15973,0,0.000000
1,2025-01-06,7,baseline_a,25458,0,0.000000
2,2025-01-13,7,baseline_a,20562,1,0.000049
3,2025-01-20,7,campaign,19744,0,0.000000
4,2025-01-27,7,campaign,11117,0,0.000000
5,2025-02-03,7,campaign,15239,0,0.000000
6,2025-02-10,7,campaign,23513,0,0.000000
7,2025-02-17,7,campaign,16919,0,0.000000
8,2025-02-24,7,post_election,12953,2,0.000154
9,2025-03-03,7,post_election,16882,4,0.000237


Weekly Germany-only volume ranges from 11,117 to 324,808 statements; three weeks exceed 200,000 (7 April: 294,298; 19 May: 217,236; 26 May: 324,808). The category is absent in 13 of the 27 weeks, including all five campaign weeks, and reaches its weekly maximum of 120 statements in the week of 31 March.

### Table 14c: content types of the Germany-only statements

Composition of all Germany-only statements (denominator: all statements of the subset) and of its
statements in the civic-discourse and elections category (denominator: all category statements of the subset). A statement listing several
content types is counted once per type.

In [30]:
cx = de.copy()
cx["content_type"] = cx["content_type"].map(lambda v: list(v) if hasattr(v, "__len__") else ["(none)"])
cx = cx.explode("content_type")
t14c = cx.groupby("content_type", as_index=False).agg(n=("n", "sum"), n_election=("n_el", "sum"))
t14c["subset_total"] = int(de["n"].sum())
t14c["share_of_subset"] = (t14c["n"] / t14c["subset_total"]).round(6)
t14c["category_total"] = int(de["n_el"].sum())
t14c["share_of_category"] = (t14c["n_election"] / t14c["category_total"]).round(6)
freeze(t14c.sort_values("n", ascending=False), "t14c_de_only_content_types")

,content_type,n,n_election,subset_total,share_of_subset,category_total,share_of_category
2,CONTENT_TYPE_OTHER,1032535,209,1595276,0.647245,243,0.860082
5,CONTENT_TYPE_VIDEO,552239,23,1595276,0.346171,243,0.094650
4,CONTENT_TYPE_TEXT,8730,9,1595276,0.005472,243,0.037037
1,CONTENT_TYPE_IMAGE,1729,2,1595276,0.001084,243,0.008230
3,CONTENT_TYPE_PRODUCT,36,0,1595276,0.000023,243,0.000000
0,CONTENT_TYPE_AUDIO,10,0,1595276,0.000006,243,0.000000


Of the 1,595,276 Germany-only statements, 64.72% concern the content type `OTHER`, 34.62% video, 0.55% text and 0.11% image. Of the subset's 243 statements in the civic-discourse and elections category, 209 (86.01%) concern `OTHER`, 23 video, 9 text and 2 image.

### Table 14d: fully automated decisions per period, Germany-only statements

In [31]:
t14d = de_w.groupby("period").agg(n_total=("n", "sum"), n_fully_automated=("n_fully", "sum")).reset_index()
t14d["period"] = pd.Categorical(t14d["period"], PERIOD_NAMES, ordered=True)
t14d = t14d.sort_values("period")
t14d["share_fully_automated"] = (t14d["n_fully_automated"] / t14d["n_total"]).round(6)
freeze(t14d, "t14d_de_only_automation")

,period,n_total,n_fully_automated,share_fully_automated
0,baseline_a,61993,61295,0.988741
2,campaign,86532,81439,0.941143
3,post_election,79027,73140,0.925506
4,transition,443415,111206,0.250794
1,baseline_b,924309,909327,0.983791


The share of fully automated decisions among Germany-only statements is 98.87% in the first baseline (61,993 statements), 94.11% during the campaign (86,532), 92.55% after the election (79,027), 25.08% in the transition (443,415) and 98.38% in the second baseline (924,309).

## Robustness: independently defined campaign window

The periods of table 6 were set after inspecting the data: the campaign period begins on 20 January 2025,
the week in which the share of the civic-discourse and elections category rises (table 5). A window
chosen from the outcome cannot carry the comparison on its own. Table 6b therefore adds a campaign window
defined independently of the data and anchored in the electoral calendar, from the dissolution of the
Bundestag on 27 December 2024 [CITE: source needed] to polling day on 23 February 2025. Within the data,
which begin on 2025-01-01, this window covers 2025-01-01 to 2025-02-23, the union of the first baseline
and the campaign period of table 6. The post-election period and the second baseline are unchanged. The
periods of table 6 are contiguous sets of days, so the counts of the combined window are the exact sums
of the two periods it spans. Both tables are computed from the frozen tables 6 and 9; no data are
scanned.

### Table 6b: period summary with the independently defined campaign window, TikTok

`rel_to_baseline_b` divides each share by the share of the second baseline; `pp_diff_to_baseline_b` is
the difference in percentage points.

In [32]:
t6_frozen = pd.read_csv(GOLD_OUT / "t06_period_summary.csv")
tt = t6_frozen[t6_frozen.platform == "TikTok"].set_index("period")
WINDOWS_6B = [
    ("campaign_independent", ["baseline_a", "campaign"], "2025-01-01", "2025-02-23"),
    ("post_election", ["post_election"], "2025-02-24", "2025-03-31"),
    ("baseline_b", ["baseline_b"], "2025-05-01", "2025-06-30"),
]
t6b = pd.DataFrame([
    {"period": name, "start": a, "end": b, "days": int(tt.loc[parts, "days"].sum()),
     "n_total": int(tt.loc[parts, "n_total"].sum()), "n_election": int(tt.loc[parts, "n_election"].sum())}
    for name, parts, a, b in WINDOWS_6B
])
share = t6b["n_election"] / t6b["n_total"]
ref_share = share[t6b["period"] == "baseline_b"].iloc[0]
t6b["share_election"] = share.round(6)
t6b["rel_to_baseline_b"] = (share / ref_share).round(4)
t6b["pp_diff_to_baseline_b"] = ((share - ref_share) * 100).round(3)
freeze(t6b, "t06b_independent_campaign_window")

,period,start,end,days,n_total,n_election,share_election,rel_to_baseline_b,pp_diff_to_baseline_b
0,campaign_independent,2025-01-01,2025-02-23,54,227211999,12281270,0.054052,1.8218,2.438
1,post_election,2025-02-24,2025-03-31,36,230517382,13232573,0.057404,1.9347,2.773
2,baseline_b,2025-05-01,2025-06-30,61,175692625,5212782,0.029670,1.0000,0.000


In the independently defined campaign window (2025-01-01 to 2025-02-23), 12,281,270 of 227,211,999 TikTok statements (5.405%) fall into the civic-discourse and elections category, against 13,232,573 of 230,517,382 (5.740%) after the election and 5,212,782 of 175,692,625 (2.967%) in the second baseline. Relative to the second baseline, the share is 1.82 times as high (+2.44 percentage points) in the campaign window and 1.93 times as high (+2.77 percentage points) after the election.

### Table 6c: daily share of the category, TikTok, 2025-01-13 to 2025-01-27

Taken from the daily series of table 9; the denominator is all TikTok statements decided on the day.

In [33]:
t9_frozen = pd.read_csv(GOLD_OUT / "t09_tiktok_daily_jan_mar.csv", parse_dates=["date"])
t6c = t9_frozen[(t9_frozen["date"] >= "2025-01-13") & (t9_frozen["date"] <= "2025-01-27")]
t6c = t6c[["date", "period", "n_total", "n_election", "share_election"]].reset_index(drop=True)
t6c["date"] = t6c["date"].dt.date
freeze(t6c, "t06c_daily_share_jan_13_27")

,date,period,n_total,n_election,share_election
0,2025-01-13,baseline_a,2527193,79615,0.031503
1,2025-01-14,baseline_a,2621375,99908,0.038113
2,2025-01-15,baseline_a,2648343,87291,0.032961
3,2025-01-16,baseline_a,2635709,71311,0.027056
4,2025-01-17,baseline_a,2798521,73998,0.026442
5,2025-01-18,baseline_a,3120751,77402,0.024802
6,2025-01-19,baseline_a,3178486,73836,0.023230
7,2025-01-20,campaign,2807873,117639,0.041896
8,2025-01-21,campaign,2810499,134267,0.047773
9,2025-01-22,campaign,3216835,174095,0.054120


From 13 to 19 January 2025 the daily share of the category lies between 2.32% and 3.81%. On 20 January it is 4.19% (117,639 of 2,807,873 statements); from 22 January onwards it lies between 5.41% and 7.65%, with the maximum on 24 January (412,098 of 5,387,564). Daily volume rises from 2,807,873 statements on 20 January to 5,387,564 on 24 January.